In [12]:
import pandas as pd
import commons as c
import plotly.express as px
import plotly.graph_objects as go
import scipy.stats as stats
import numpy as np


# Get datasets

In [2]:
csv_path = 'results/dataframes/results_all_mutants.csv'
df = pd.read_csv(csv_path, dtype=c.type_dict)

# Get Box Plots

In [3]:
category_names = {
    'Qubits_number': 'Number of qubits', 
    'gates': 'Number of gates', 
    'depth': 'Circuit depth',
    'Algorithm': 'Algorithm name', 
    'Input_type': 'Type of input', 
    'Output_type': 'Type of output',
    'Gate_type': 'Mutated gate', 
    'Operator': 'Mutation operator', 
    'Relative_position': 'Relative position of the mutation'
}

In [4]:
def print_box_plot(df, cat, distance, output_folder, file_name, median_line=False, width=2000):
    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column   
    
    # Create the box plot
    fig = px.box(
        df, 
        y=distance, 
        x=cat, 
        color="nature", 
        category_orders={
            cat: cat_range,
            "nature": ["Equivalent mutant", "Non-Equivalent mutant"]
        },
        # title="Boxplot of Distance by noise model and program type",
        labels={cat: category_names[cat], distance: "Distance", 'nature': "Legend"},
        points=False,
        boxmode="group"
    ) 
    
    # Compute medians for each category and true_label
    median_values = df.groupby([cat, "nature"])[distance].median().reset_index()
    
    if median_line:
        # Define colors matching the boxplot
        colors = {"Equivalent mutant": "blue", "Non-Equivalent mutant": "red"}
        
        for nature in median_values["nature"].unique():
            subset = median_values[median_values["nature"] == nature]
            fig.add_trace(go.Scatter(
                x=subset[cat], 
                y=subset[distance], 
                mode='lines',
                name=f"Median - {nature}",
                line=dict(color=colors[nature]) #, dash='dot')
            ))
    
    # Adjust layout for better visualization
    fig.update_layout(
        xaxis=dict(tickmode="array", tickvals=cat_range)
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1], width=width)

In [5]:
def category_plot(df, m):
    category_groups = [
        ('Qubits_number', "RQ2_1", True, 3000), # 7
        ('gates', "RQ2_1", True, 3000), # 65
        ('depth', "RQ2_1", True, 3000), # 42
        ('Algorithm', "RQ2_2", False, 2000), # 5
        ('Input_type', "RQ2_2", False, 1000), # 2
        ('Output_type', "RQ2_2", False, 1000), # 2
        ('Gate_type', "RQ2_3", False, 1000), # 2
        ('Operator', "RQ2_3", False, 1000), # 3
        ('Relative_position', "RQ2_3", True, 2000) # 5
    ]
    
    for hw in c.hardware:
        df_hw = df[df['hardware'] == hw]
        df_metric = df_hw[df_hw['metric'] == m]

        for cat, subfolder, showmedian, width in category_groups:
            selected_columns = df_metric[[cat, 'nature', 'ideal_distance', 'noisy_distance']]
            file_name = f'{hw}_{cat}'
                
            for distance_type in ['noisy_distance', 'ideal_distance']:
                variant = 'noisy' if distance_type == 'noisy_distance' else 'ideal'
                output_folder = f'results/RQ2/{subfolder}/{variant}/{m}'
                print_box_plot(selected_columns, cat, distance_type, output_folder, file_name, showmedian, width)


In [ ]:
m = "T"
category_plot(df, m)

m = "H"
category_plot(df, m)

# Statistical tests

In [6]:
df

,gates,depth,singlequbit_gates,multiqubit_gates,Input,Input_type,Algorithm,Qubits_number,Operator,Gate,...,nature,metric,metric_full,threshold,ideal_distance,noisy_distance,ideal_label,noisy_label,correctness,hardware_named
0,8,6,5,3,PureState_0,PureState,ae,2,Add,ch,...,Equivalent mutant,H,Hellinger,I,4.359133e-03,0.017119,False,False,True,Kyiv noise model
1,8,6,5,3,Quratest_0,Quratest,ae,2,Add,ch,...,Equivalent mutant,H,Hellinger,I,4.100506e-03,0.019849,False,False,True,Kyiv noise model
2,8,6,5,3,PureState_1,PureState,ae,2,Add,ch,...,Equivalent mutant,H,Hellinger,I,8.032655e-03,0.017959,False,False,True,Kyiv noise model
3,8,6,5,3,Quratest_1,Quratest,ae,2,Add,ch,...,Equivalent mutant,H,Hellinger,I,2.416967e-03,0.044957,False,False,True,Kyiv noise model
4,8,6,5,3,PureState_2,PureState,ae,2,Add,ch,...,Equivalent mutant,H,Hellinger,I,6.913256e-03,0.022281,False,False,True,Kyiv noise model
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13821115,30,17,15,15,Quratest_125,Quratest,wstate,8,Replace,z,...,Non-Equivalent mutant,E,Expectation Values,A,1.124124e-02,0.028819,False,False,True,Sherbrooke noise model
13821116,30,17,15,15,PureState_126,PureState,wstate,8,Replace,z,...,Non-Equivalent mutant,E,Expectation Values,A,2.220446e-16,0.515625,False,True,False,Sherbrooke noise model
13821117,30,17,15,15,Quratest_126,Quratest,wstate,8,Replace,z,...,Non-Equivalent mutant,E,Expectation Values,A,6.178777e-03,0.352858,False,True,False,Sherbrooke noise model
13821118,30,17,15,15,PureState_127,PureState,wstate,8,Replace,z,...,Non-Equivalent mutant,E,Expectation Values,A,4.440892e-16,0.518066,False,True,False,Sherbrooke noise model


In [7]:
for metric in df['metric'].unique():
    subset_df = df[df['metric'] == metric]
    df_equiv = subset_df[subset_df['nature']=='Equivalent mutant']
    df_non_equiv = subset_df[subset_df['nature']=='Non-Equivalent mutant']
    
    numeric_df_equiv = df_equiv[['Qubits_number','depth','gates','singlequbit_gates','multiqubit_gates','ideal_distance','noisy_distance']]
    numeric_df_non_equiv = df_non_equiv[['Qubits_number','depth','gates','singlequbit_gates','multiqubit_gates','ideal_distance','noisy_distance']]
    correlation_equiv = numeric_df_equiv.corr(method='pearson')[['ideal_distance','noisy_distance']]
    correlation_equiv = correlation_equiv.drop(['ideal_distance','noisy_distance'])
    print(f'\nEquiv mutants metric {metric}')
    print(correlation_equiv)
    
    correlation_non_equiv = numeric_df_non_equiv.corr(method='pearson')[['ideal_distance','noisy_distance']]
    correlation_non_equiv = correlation_non_equiv.drop(['ideal_distance','noisy_distance'])
    print(f'\nNon-Equiv mutants metric {metric}')
    print(correlation_non_equiv)


Equiv mutants metric H
                   ideal_distance  noisy_distance
Qubits_number            0.050542        0.133159
depth                   -0.024029       -0.066813
gates                    0.013681       -0.011879
singlequbit_gates       -0.093528       -0.003518
multiqubit_gates         0.089006       -0.012899

Non-Equiv mutants metric H
                   ideal_distance  noisy_distance
Qubits_number            0.045136        0.076299
depth                   -0.088671       -0.102182
gates                   -0.056960       -0.068316
singlequbit_gates        0.094039        0.129448
multiqubit_gates        -0.145333       -0.186799

Equiv mutants metric J
                   ideal_distance  noisy_distance
Qubits_number            0.054361        0.144653
depth                   -0.021330       -0.052822
gates                    0.018208        0.005314
singlequbit_gates       -0.094775        0.002796
multiqubit_gates         0.095888        0.004843

Non-Equiv mutants metri

In [13]:

def pearson_correlation_pvalues(df):
    cols = df.select_dtypes(include=[np.number]).columns  # Select numeric columns
    n = len(cols)

    corr_matrix = np.zeros((n, n))
    pval_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i == j:
                corr_matrix[i, j] = 1.0  # Diagonal elements (self-correlation)
                pval_matrix[i, j] = 0.0  # No p-value for self-correlation
            else:
                r, p = stats.pearsonr(df[cols[i]], df[cols[j]])  # Compute correlation & p-value
                corr_matrix[i, j] = r
                pval_matrix[i, j] = p

    return pd.DataFrame(corr_matrix, index=cols, columns=cols), pd.DataFrame(pval_matrix, index=cols, columns=cols)

for metric in df['metric'].unique():
    subset_df = df[df['metric'] == metric]
    df_equiv = subset_df[subset_df['nature']=='Equivalent mutant']
    df_non_equiv = subset_df[subset_df['nature']=='Non-Equivalent mutant']
    numeric_df_equiv = df_equiv[['Qubits_number','depth','gates','singlequbit_gates','multiqubit_gates','ideal_distance','noisy_distance']]
    numeric_df_non_equiv = df_non_equiv[['Qubits_number','depth','gates','singlequbit_gates','multiqubit_gates','ideal_distance','noisy_distance']]
    correlation_matrix_equiv, p_value_matrix_equiv = pearson_correlation_pvalues(numeric_df_equiv)
    correlation_matrix_non_equiv, p_value_matrix_non_equiv = pearson_correlation_pvalues(numeric_df_non_equiv)

    print(f'\n Correlation equiv metric {metric}')
    print(correlation_matrix_equiv)
    print(f'\n P-Values equiv metric {metric}')
    print(p_value_matrix_equiv)
    print(f'\n Correlation non-equiv metric {metric}')
    print(correlation_matrix_non_equiv)
    print(f'\n P-Values non-equiv metric {metric}')
    print(p_value_matrix_non_equiv)


 Correlation equiv metric H
                   Qubits_number     depth     gates  singlequbit_gates  \
Qubits_number           1.000000  0.366762  0.684382           0.350435   
depth                   0.366762  1.000000  0.871212           0.720524   
gates                   0.684382  0.871212  1.000000           0.650493   
singlequbit_gates       0.350435  0.720524  0.650493           1.000000   
multiqubit_gates        0.630894  0.594591  0.816640           0.092868   
ideal_distance          0.050542 -0.024029  0.013681          -0.093528   
noisy_distance          0.133159 -0.066813 -0.011879          -0.003518   

                   multiqubit_gates  ideal_distance  noisy_distance  
Qubits_number              0.630894        0.050542        0.133159  
depth                      0.594591       -0.024029       -0.066813  
gates                      0.816640        0.013681       -0.011879  
singlequbit_gates          0.092868       -0.093528       -0.003518  
multiqubit_gates    

In [10]:
categorical_columns = ['Algorithm','Relative_position','Output_type','Input_type', 'Gate_type','Operator']
numerical_columns = ['ideal_distance','noisy_distance']

for metric in df['metric'].unique():
    subset_df = df[df['metric'] == metric]
    df_equiv = subset_df[subset_df['nature']=='Equivalent mutant']
    df_non_equiv = subset_df[subset_df['nature']=='Non-Equivalent mutant']
    results_equiv = {}
    for cat_col in categorical_columns:
        result_row = {}
        for num_col in numerical_columns:
            groups = [df_equiv[df_equiv[cat_col] == val][num_col].dropna() for val in df_equiv[cat_col].unique()]
            
            if len(groups) < 2 or any(len(g) == 0 for g in groups):
                f_stat, p_val = np.nan, np.nan
            else:
                f_stat, p_val = stats.f_oneway(*groups)
            
            result_row[f'{num_col}_F'] = f_stat
            result_row[f'{num_col}_p'] = p_val
            
            # Add significance interpretation
            if pd.notna(p_val):
                result_row[f'{num_col}_significant'] = "Significant" if p_val < 0.05 else "Not Significant"
            else:
                result_row[f'{num_col}_significant'] = "N/A"
    
        results_equiv[cat_col] = result_row
    
    # Create DataFrame
    anova_df_equiv = pd.DataFrame.from_dict(results_equiv, orient='index')
    print(f'\nEquiv mutants metric {metric}')
    print(anova_df_equiv)
    
    results_non_equiv = {}
    for cat_col in categorical_columns:
        result_row = {}
        for num_col in numerical_columns:
            groups = [df_non_equiv[df_non_equiv[cat_col] == val][num_col].dropna() for val in df_non_equiv[cat_col].unique()]
            
            if len(groups) < 2 or any(len(g) == 0 for g in groups):
                f_stat, p_val = np.nan, np.nan
            else:
                f_stat, p_val = stats.f_oneway(*groups)
            
            result_row[f'{num_col}_F'] = f_stat
            result_row[f'{num_col}_p'] = p_val
            
            # Add significance interpretation
            if pd.notna(p_val):
                result_row[f'{num_col}_significant'] = "Significant" if p_val < 0.05 else "Not Significant"
            else:
                result_row[f'{num_col}_significant'] = "N/A"
    
        results_non_equiv[cat_col] = result_row
    
    # Create DataFrame
    anova_df_non_equiv = pd.DataFrame.from_dict(results_non_equiv, orient='index')
    print(f'\nNon-Equiv mutants metric {metric}')
    print(anova_df_non_equiv)



Equiv mutants metric H
                   ideal_distance_F  ideal_distance_p  \
Algorithm             423837.045705      0.000000e+00   
Relative_position         41.726802      4.845868e-35   
Output_type           142168.577571      0.000000e+00   
Input_type              7839.750010      0.000000e+00   
Gate_type                  0.611157      4.343527e-01   
Operator                        NaN               NaN   

                  ideal_distance_significant  noisy_distance_F  \
Algorithm                        Significant     458761.538786   
Relative_position                Significant        115.008385   
Output_type                      Significant     169912.759911   
Input_type                       Significant       3725.281138   
Gate_type                    Not Significant       1320.586060   
Operator                                 N/A               NaN   

                   noisy_distance_p noisy_distance_significant  
Algorithm              0.000000e+00             

In [11]:
categorical_columns = ['Algorithm','Relative_position','Output_type','Input_type', 'Gate_type','Operator']
numerical_columns = ['ideal_distance','noisy_distance']

for metric in df['metric'].unique():
    subset_df = df[df['metric'] == metric]
    df_equiv = subset_df[subset_df['nature']=='Equivalent mutant']
    df_non_equiv = subset_df[subset_df['nature']=='Non-Equivalent mutant']
    # Initialize results dictionary
    results_equiv = {}
    for cat_col in categorical_columns:
        result_row = {}
        for num_col in numerical_columns:
            groups = [df_equiv[df_equiv[cat_col] == val][num_col].dropna() for val in df_equiv[cat_col].unique()]
            
            if len(groups) < 2 or any(len(g) == 0 for g in groups):
                H_stat, p_val = np.nan, np.nan
            else:
                H_stat, p_val = stats.kruskal(*groups)
            
            result_row[f'{num_col}_H'] = H_stat
            result_row[f'{num_col}_p'] = p_val
            
            # Add significance interpretation
            if pd.notna(p_val):
                result_row[f'{num_col}_significant'] = "Significant" if p_val < 0.05 else "Not Significant"
            else:
                result_row[f'{num_col}_significant'] = "N/A"
    
        results_equiv[cat_col] = result_row
    
    # Create DataFrame
    kruskal_df_equiv = pd.DataFrame.from_dict(results_equiv, orient='index')
    print(f'Equiv mutants metric {metric}')
    print(kruskal_df_equiv)
    
    results_non_equiv = {}
    for cat_col in categorical_columns:
        result_row = {}
        for num_col in numerical_columns:
            groups = [df_non_equiv[df_non_equiv[cat_col] == val][num_col].dropna() for val in df_non_equiv[cat_col].unique()]
            
            if len(groups) < 2 or any(len(g) == 0 for g in groups):
                H_stat, p_val = np.nan, np.nan
            else:
                H_stat, p_val = stats.kruskal(*groups)
            
            result_row[f'{num_col}_H'] = H_stat
            result_row[f'{num_col}_p'] = p_val
            
            # Add significance interpretation
            if pd.notna(p_val):
                result_row[f'{num_col}_significant'] = "Significant" if p_val < 0.05 else "Not Significant"
            else:
                result_row[f'{num_col}_significant'] = "N/A"
    
        results_non_equiv[cat_col] = result_row
    
    # Create DataFrame
    kruskal_df_non_equiv = pd.DataFrame.from_dict(results_non_equiv, orient='index')
    print(f'\nNon-Equiv mutants metric {metric}')
    print(kruskal_df_non_equiv)


Equiv mutants metric H
                   ideal_distance_H  ideal_distance_p  \
Algorithm             369439.358160      0.000000e+00   
Relative_position       1007.790568     7.315770e-217   
Output_type            29722.321411      0.000000e+00   
Input_type            134739.260062      0.000000e+00   
Gate_type              14800.708754      0.000000e+00   
Operator                        NaN               NaN   

                  ideal_distance_significant  noisy_distance_H  \
Algorithm                        Significant     728936.636112   
Relative_position                Significant       1656.487670   
Output_type                      Significant     118086.180851   
Input_type                       Significant      27151.461723   
Gate_type                        Significant       3043.920292   
Operator                                 N/A               NaN   

                   noisy_distance_p noisy_distance_significant  
Algorithm                       0.0              